In [2]:
# !pip install torch torchvision matplotlib


In [1]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [2]:
BATCH = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor()),
                     batch_size=BATCH, shuffle=True)


In [3]:
class SparseAE(nn.Module):
    def __init__(self, rho=0.05, beta=3):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 1000), nn.Sigmoid())
        self.dec = nn.Linear(1000, 784)
        self.rho, self.beta = rho, beta
    def forward(self, x):
        z = self.enc(x.view(x.size(0), -1))
        return self.dec(z), z
    def kl(self, z):
        rho_hat = z.mean(0)
        return self.beta * (self.rho * torch.log(self.rho / (rho_hat + 1e-8)) + (1 - self.rho) * torch.log((1 - self.rho) / (1 - rho_hat + 1e-8))).sum()


In [4]:
model = SparseAE().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(5):
    for x, _ in loader:
        x = x.to(device)
        recon, z = model(x)
        loss = nn.MSELoss()(recon, x.view(x.size(0), -1)) + model.kl(z)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f'epoch {epoch+1} loss {loss.item():.4f}')


epoch 1 loss 0.1087


epoch 2 loss 1.0253


epoch 3 loss 1.0378


epoch 4 loss 0.0651


epoch 5 loss 1.0433


In [6]:
model.eval()
with torch.no_grad():
    x, _ = next(iter(loader))
    x = x.to(device)
    recon, z = model(x)
    print('mean activation (sparsity):', z.mean(0)[:10].cpu().numpy())


mean activation (sparsity): [0.04886235 0.04989957 0.05005686 0.05058032 0.05021238 0.04952115
 0.05008623 0.04981178 0.05084112 0.04918691]
